In [9]:
import os
import yaml
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

# Paths relative to notebooks/
ROOT_DIR = os.path.abspath(os.path.join(".."))
CFG_PATH = os.path.join(ROOT_DIR, "src", "config.yaml")

with open(CFG_PATH, encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

CFG.get("c1_tariffs", {})

{'user_agent': 'ULSmartMeterFYP/0.1 (student final year project, University of Limerick)',
 'suppliers': [{'name': 'Electric Ireland',
   'url': 'https://www.electricireland.ie/switch/new-customer/price-plans?priceType=E',
   'method': 'static'}]}

In [10]:
tar_cfg = CFG["c1_tariffs"]
sup = tar_cfg["suppliers"][0]
URL = sup["url"]
UA = tar_cfg["user_agent"]

print("Target supplier:", sup["name"])
print("URL:", URL)

# Create a polite session (recommended)
session = requests.Session()
session.headers.update({
    "User-Agent": UA,
    "Accept-Language": "en-IE,en;q=0.9",
})

resp = session.get(URL, timeout=30)
print("Status code:", resp.status_code)

# Show a small slice of HTML so we know we got the page
print(resp.text[:1000])

Target supplier: Electric Ireland
URL: https://www.electricireland.ie/switch/new-customer/price-plans?priceType=E
Status code: 200

<!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js lt-ie9 lt-ie8 lt-ie7" lang="en"> <![endif]-->
<!--[if IE 7]> <html class="no-js lt-ie9 lt-ie8" lang="en"> <![endif]-->
<!--[if IE 8]> <html class="no-js lt-ie9" lang="en"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en">
 <!--<![endif]-->

<head>
        <meta name="robots" content="all">

    <meta http-equiv="X-UA-Compatible" content="IE=edge" />
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Switch electricity price plans today | Electric Ireland </title>
    <meta name="description" content="Switch electricity to one of our price plans and see how much you can save today. We’ll help you find our cheapest electricity rates.">

    <link rel="preconnect" href="//www.googletagmanager.com">
    <link rel="preconnect" href

In [11]:
from datetime import datetime, timezone

RAW_DIR = os.path.join(ROOT_DIR, "data", "tariffs", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
raw_path = os.path.join(RAW_DIR, f"{sup['name'].lower()}_{ts}.html")

with open(raw_path, "w", encoding="utf-8") as f:
    f.write(resp.text)

print("Saved raw HTML to:", raw_path)

Saved raw HTML to: C:\Users\eoinp\SmartMeterProject\data\tariffs\raw\electric ireland_20260125_2009.html


In [12]:
from bs4 import BeautifulSoup

# Load the saved HTML
with open(raw_path, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

# Search for phrases we know appear on the page
search_terms = [
    "Electricity unit price",
    "Day:",
    "Night:",
    "Peak:",
    "First year cost",
    "Estimated Annual Bill"
]

for term in search_terms:
    results = soup.find_all(string=lambda t: term in t)
    print(f"\n=== {term} — found {len(results)} matches ===")
    for r in results[:2]:
        print("Context:", r.parent.name, "|", r.strip())
        print(r.parent.decode()[:300], "...")


=== Electricity unit price — found 9 matches ===
Context: td | Electricity unit price
<td>Electricity unit price</td> ...
Context: td | Electricity unit price
<td colspan="2">Electricity unit price</td> ...

=== Day: — found 5 matches ===
Context: td | Day: 08.00 - 23.00
<td>Day: 08.00 - 23.00</td> ...
Context: td | Day: 08.00 - 23.00
<td>Day: 08.00 - 23.00</td> ...

=== Night: — found 5 matches ===
Context: td | Night: 23.00 - 08.00
<td>Night: 23.00 - 08.00</td> ...
Context: td | Night: 23.00 - 08.00
<td>Night: 23.00 - 08.00</td> ...

=== Peak: — found 2 matches ===
Context: td | Peak: 17.00 - 19.00
<td>Peak: 17.00 - 19.00</td> ...
Context: td | Peak: 17.00 - 19.00
<td>Peak: 17.00 - 19.00</td> ...

=== First year cost — found 9 matches ===
Context: td | First year cost
<td>First year cost</td> ...
Context: td | First year cost
<td>First year cost</td> ...

=== Estimated Annual Bill — found 9 matches ===
Context: div | Estimated Annual Bill (EAB)
<div>

                               

In [13]:
import os
import re
from datetime import datetime, timezone
import pandas as pd
from bs4 import BeautifulSoup

# 1) Load the saved HTML again (raw_path was created in the earlier cell)
with open(raw_path, "r", encoding="utf-8") as f:
    html = f.read()

# Use built-in parser to avoid lxml dependency issues
soup = BeautifulSoup(html, "html.parser")

# 2) Find all plan "cards"
cards = soup.select("div#scrollable div.scrollable-card")
print(f"Found {len(cards)} plan cards total")

def extract_cent_price(cell_text: str):
    """
    Take text like '28.14c per kWh' and return 0.2814 (euro/kWh).
    """
    if not cell_text:
        return None
    m = re.search(r"([\d,.]+)\s*c", cell_text)
    if not m:
        return None
    val = m.group(1).replace(",", "")
    try:
        return float(val) / 100.0
    except ValueError:
        return None

def extract_euro(cell_text: str):
    """
    Take text like '€1,223' and return 1223.0
    """
    if not cell_text:
        return None
    m = re.search(r"€\s*([\d,]+(?:\.\d+)?)", cell_text)
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return None

def parse_card(card):
    # --- Plan name ---
    h2 = card.select_one("h2")
    plan_name = h2.get_text(strip=True) if h2 else "Unknown"

    # --- Meter type ---
    meter_type_raw = (card.get("data-meter-type") or "").strip()
    meter_type_map = {
        "Smart": "Smart",
        "Standard": "Standard",
        "NightSaver": "DayNight",
        "DayNight": "DayNight",
    }
    meter_type = meter_type_map.get(meter_type_raw, meter_type_raw or "Unknown")

    # --- Price table ---
    table = card.select_one("table.prices")
    if table is None:
        return {
            "supplier": sup["name"],
            "plan_name": plan_name,
            "meter_type": meter_type,
            "structure": "Unknown",
            "unit_rate_24h_eur_kwh": None,
            "unit_rate_day_eur_kwh": None,
            "unit_rate_night_eur_kwh": None,
            "unit_rate_peak_eur_kwh": None,
            "first_year_cost_eur": None,
            "estimated_annual_bill_eur": None,
            "source_url": sup["url"],
            "scraped_at_utc": datetime.now(timezone.utc).isoformat(),
        }

    unit_24 = unit_day = unit_night = unit_peak = None
    first_year = None

    # --- NEW: try to detect a single-rate ("24h") price anywhere in the card text ---
    # This catches plans like "Electricity unit price 27.80c per kWh" which often
    # appear outside the Day/Night/Peak rows.
    card_text = card.get_text(" ", strip=True)
    m = re.search(r"Electricity unit price\s*([\d,.]+)\s*c", card_text, flags=re.IGNORECASE)
    if m:
        unit_24 = extract_cent_price(m.group(1) + "c")  # reuse your helper safely

    for row in table.select("tr"):
        tds = row.find_all("td")
        if not tds:
            continue
        label = tds[0].get_text(" ", strip=True)
        value = tds[-1].get_text(" ", strip=True)

        if "Electricity unit price" in label:
            continue

        if label.startswith("Day:"):
            unit_day = extract_cent_price(value)
        elif label.startswith("Night:"):
            unit_night = extract_cent_price(value)
        elif label.startswith("Peak:"):
            unit_peak = extract_cent_price(value)
        elif label.startswith("First year cost"):
            first_year = extract_euro(value)
        elif "per kWh" in value and not any(x in label for x in ["Day", "Night", "Peak"]):
            unit_24 = extract_cent_price(value)


    # --- Estimated Annual Bill (EAB) ---
    eab = None
    spread = card.select_one("ul.spread")
    if spread:
        for li in spread.select("li"):
            text = li.get_text(" ", strip=True)
            if "Estimated Annual Bill" in text:
                eab = extract_euro(text)
                break

    # --- Structure classification ---
    has_24 = unit_24 is not None
    has_day = unit_day is not None
    has_night = unit_night is not None
    has_peak = unit_peak is not None

    if has_day and has_night and has_peak:
        structure = "3-Period"
    elif has_day and has_night:
        structure = "DayNight"
    elif has_24:
        structure = "24h"
    else:
        structure = "24h"  # fallback: single-rate plans


    return {
        "supplier": sup["name"],
        "plan_name": plan_name,
        "meter_type": meter_type,
        "structure": structure,
        "unit_rate_24h_eur_kwh": unit_24,
        "unit_rate_day_eur_kwh": unit_day,
        "unit_rate_night_eur_kwh": unit_night,
        "unit_rate_peak_eur_kwh": unit_peak,
        "first_year_cost_eur": first_year,
        "estimated_annual_bill_eur": eab,
        "source_url": sup["url"],
        "scraped_at_utc": datetime.now(timezone.utc).isoformat(),
    }

records = [parse_card(c) for c in cards]
df_tariffs = pd.DataFrame.from_records(records)

print("Rows parsed:", len(df_tariffs))
display(df_tariffs)




Found 9 plan cards total
Rows parsed: 9


,supplier,plan_name,meter_type,structure,unit_rate_24h_eur_kwh,unit_rate_day_eur_kwh,unit_rate_night_eur_kwh,unit_rate_peak_eur_kwh,first_year_cost_eur,estimated_annual_bill_eur,source_url,scraped_at_utc
0,Electric Ireland,EnergySaver 20%,Standard,24h,0.2780,NaN,NaN,NaN,1337.0,1437.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.694369+00:00
1,Electric Ireland,EnergySaver NightSaver 20%,DayNight,DayNight,NaN,0.2968,0.1464,NaN,1254.0,1354.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.695370+00:00
2,Electric Ireland,Green Electricity,Standard,24h,0.3300,NaN,NaN,NaN,1656.0,1656.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.695370+00:00
3,Electric Ireland,Green Electricity NightSaver,DayNight,DayNight,NaN,0.3527,0.1749,NaN,1545.0,1545.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.696366+00:00
4,Electric Ireland,Home Electric + SST Saver 24%,Smart,3-Period,NaN,0.2814,0.1479,0.3002,1220.0,1255.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.697370+00:00
5,Electric Ireland,Home Electric + Saver 24%,Smart,24h,0.2586,NaN,NaN,NaN,1236.0,1356.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.697370+00:00
6,Electric Ireland,Home Electric + Night Boost 16%,Smart,DayNight,NaN,0.3053,0.1505,NaN,1370.0,1370.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.698311+00:00
7,Electric Ireland,Home Electric+ Weekender Saver 16%,Smart,24h,0.3138,NaN,NaN,NaN,1471.0,1471.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.698311+00:00
8,Electric Ireland,Home Electric SST Saver 24%,Smart,3-Period,NaN,0.2814,0.1479,0.3002,1220.0,1255.0,https://www.electricireland.ie/switch/new-cust...,2026-01-25T20:09:39.699306+00:00


In [15]:
import re
import os

def slugify(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

out_dir = os.path.join(ROOT_DIR, "data", "tariffs", "clean")
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, f"{slugify(sup['name'])}_all_plans.csv")
df_tariffs.to_csv(out_path, index=False, encoding="utf-8")

print("Saved cleaned tariffs to:", out_path)




Saved cleaned tariffs to: C:\Users\eoinp\SmartMeterProject\data\tariffs\clean\electric_ireland_all_plans.csv
